# Imports

In [1]:
# Add the directory where starships' directory is located
from sys import path
path.append('/home/mathisb/Github/starships')

# Add the directory containing the input data (opacity, abundance and stellar specs files)
import os
os.environ['pRT_input_data_path'] = '/home/mathisb/projects/def-rdoyon/shared/Models/petitRADTRANS/input_data/'

In [2]:
from pathlib import Path
from importlib import reload

import matplotlib.pyplot as plt
import starships.homemade as hm
from starships import spectrum as spectrum
from starships.mask_tools import interp1d_masked

interp1d_masked.iprint=False
import warnings

import astropy.constants as const
import astropy.units as u
import numpy as np
import starships.planet_obs as pl_obs
import starships.plotting_fcts as pf
from starships.planet_obs import Observations

warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", RuntimeWarning)

# Read spectra and useful quantities

## Note: Better to use the scratch space to save the reductions.
You have infinite space, but the files are deleted if untouched for 2 months. It allows to save as many reductions as desired. Once the correct reduction parameters are set, you can move them into your home directory.

In [3]:
#--------------------------------------------------------------------------------------------
# Planet, night, reduction and instrument
pl_name = 'TRAPPIST-1 g'
visit_name = '2025-07-08'
reduction = "v001"
instrument = "NIRPS"  # SPIRou_with_persi, NIRPS

reduc_in_scratch = True  # True to put the reductions in scratch, False to put them in the Github folder
#--------------------------------------------------------------------------------------------

# Directory where to save reductions
if reduc_in_scratch:
    # > use scratch if available, use home if not
    try:
        out_dir = Path(os.environ['SCRATCH'])
    except KeyError:
        out_dir = Path.home()
else:
    out_dir = Path.home() / "Github/"

# Output reductions in dedicated directory
pl_name_fname = ''.join(pl_name.split())
out_dir /= Path(f'HRS_reductions/{instrument}/{pl_name_fname}/{visit_name}/{reduction}')

# Make sure the  directory exists
out_dir.mkdir(parents=True, exist_ok=True)


In [4]:
# Relevant directories

# Where to find the observations (data)?
obs_dir = f"/home/mathisb/Github/HRS_data/{instrument}/TRAPPIST-1/{visit_name}/"  # for TRAPPIST-1 data

# Where to save figures?
path_fig = Path(f'HRS_reductions/{instrument}/{pl_name_fname}/{visit_name}/Figures')

In [5]:
# Create text files with all the e (e2ds) and t (tcorr) data files listed

# # ed2s
# with open(obs_dir + 'list_e2ds.txt', 'a+') as e_txt:
#     for i in sorted(os.listdir(obs_dir)):
#         if i[-6:] == "e.fits":
#             e_txt.write(i + '\n')

# # tcorr
# with open(obs_dir + 'list_tcorr.txt', 'a+') as t_txt:
#     for i in sorted(os.listdir(obs_dir)):
#         if i[-6:] == "t.fits":
#             t_txt.write(i + '\n')


In [6]:
# All the observations must be listed in files.
# We need the e2ds and the telluric corrected spectra.

list_filenames = {'list_e2ds': 'list_e2ds.txt',
                  'list_tcorr': 'list_tcorr.txt'}


In [7]:
# SPECIFY PARAMETERS OF THE PLANET
# If not specified, the default parameters from exofile are taken.

# # TRAPPIST-1 b
# ap     = 0.01154*u.au        # semi-major axis of planet
# R_pl   = 1.116*u.R_earth     # radius of planet
# R_star = 0.1192*u.R_sun      # radius of star
# M_star = 0.0898*const.M_sun  # mass of star
# e      = 0                   # eccentricity
# w      = 4.712389*u.rad      # argument of periapsis (equal to 270 degrees)

# # TRAPPIST-1 d
# ap     = 0.02227*u.au        # semi-major axis of planet
# R_pl   = 0.788*u.R_earth     # radius of planet
# R_star = 0.1192*u.R_sun      # radius of star
# M_star = 0.0898*const.M_sun  # mass of star
# e      = 0                   # eccentricity
# w      = 4.712389*u.rad      # argument of periapsis (equal to 270 degrees)

# # TRAPPIST-1 e
# ap     = 0.02925*u.au        # semi-major axis of planet
# R_pl   = 0.920*u.R_earth     # radius of planet
# R_star = 0.1192*u.R_sun      # radius of star
# M_star = 0.0898*const.M_sun  # mass of star
# e      = 0                   # eccentricity
# w      = 4.712389*u.rad      # argument of periapsis (equal to 270 degrees)

# # TRAPPIST-1 f
# ap     = 0.03849*u.au        # semi-major axis of planet
# R_pl   = 1.045*u.R_earth     # radius of planet
# R_star = 0.1192*u.R_sun      # radius of star
# M_star = 0.0898*const.M_sun  # mass of star
# e      = 0                   # eccentricity
# w      = 4.712389*u.rad      # argument of periapsis (equal to 270 degrees)

# TRAPPIST-1 g
ap     = 0.04683*u.au        # semi-major axis of planet
R_pl   = 1.129*u.R_earth     # radius of planet
R_star = 0.1192*u.R_sun      # radius of star
M_star = 0.0898*const.M_sun  # mass of star
e      = 0                   # eccentricity
w      = 4.712389*u.rad      # argument of periapsis (equal to 270 degrees)

#-----------------------------------------------------------#


# Mid-transit time (BJD) of each TRAPPIST-1 visit (calculated from Agol et al. 2024)
mid_tr_dict = {
        # SPIRou b
               "2019-06-14": 2458649.07123371738 * u.d,
               "2019-09-25": 2458751.8112203472  * u.d,
               "2020-05-31": 2459001.10676193183 * u.d,
               "2020-08-04": 2459066.07481311538 * u.d,
               "2020-09-08": 2459100.82490738839 * u.d,
               "2020-09-20": 2459112.91190875796 * u.d,
               "2020-09-26": 2459118.95517847751 * u.d,
               "2020-09-29": 2459121.9775049779  * u.d,
               "2021-10-21": 2459508.76251261068 * u.d,
               "2021-10-24": 2459511.78471495466 * u.d,
               "2021-10-27": 2459514.8059938763  * u.d,
        # NIRPS b
               "2023-09-07": 2460194.7007329288  * u.d,
               "2024-07-05": 2460496.8725766301  * u.d,
               "2024-07-11b": 2460502.9159761349 * u.d,
        # NIRPS d
               "2023-06-08": 2460103.8951073092 * u.d,
               "2023-08-12": 2460168.6902942523 * u.d,
               "2023-08-24": 2460180.8397053848 * u.d,
               "2023-10-20": 2460237.5365671548 * u.d,
        # NIRPS e
               "2023-07-17": 2460142.858565431  * u.d,
               "2023-11-04": 2460252.6376593957 * u.d,
               "2024-07-11e": 2460502.7365562134 * u.d,
        # NIRPS f
               "2023-08-28": 2460184.827486578  * u.d,
               "2024-11-02": 2460617.5481359637 * u.d,
               "2025-08-06": 2460893.795434583  * u.d,
        # NIRPS g
               "2023-10-05": 2460222.5638302773 * u.d,
               "2023-11-11": 2460259.6204310769 * u.d,
               "2025-07-08": 2460864.8974261757 * u.d,
                }

mid_tr = mid_tr_dict[visit_name]

if 'SPIRou' in instrument: instr_name = 'SPIRou-APERO'
elif 'NIRPS' in instrument: instr_name = 'NIRPS-APERO'

obs = Observations(name=pl_name, instrument=instr_name,
                   pl_kwargs = {'M_star': M_star, 
                                'R_star': R_star,
                                'ap': ap,
                                'R_pl': R_pl,
                                'mid_tr': mid_tr,
                                't_peri': mid_tr,
                                'excent': e,
                                'w': w})

# Get the data
obs.fetch_data(obs_dir, **list_filenames, CADC=True)
p = obs.planet

INFO:starships.planet_obs:Getting TRAPPIST-1 g from ExoFile
INFO:starships.planet_obs:Changing M_star from [1.5907279e+29] kg to 1.78559206388685e+29 kg
INFO:starships.planet_obs:It became [1.78559206e+29] kg
INFO:starships.planet_obs:Changing R_star from [0.] m to 0.1192 solRad
INFO:starships.planet_obs:It became [82927440.] m
INFO:starships.planet_obs:Changing ap from [7.00342432e+09] m to 0.04683 AU
INFO:starships.planet_obs:It became [7.00566828e+09] m
INFO:starships.planet_obs:Changing R_pl from [7220692.] m to 1.129 earthRad
INFO:starships.planet_obs:It became [7200874.9] m
INFO:starships.planet_obs:Changing mid_tr from [2457665.362844] d to 2460864.897426176 d
INFO:starships.planet_obs:It became [2460864.89742618] d
INFO:starships.planet_obs:Changing t_peri from [2457665.362844] d to 2460864.897426176 d
INFO:starships.planet_obs:It became [2460864.89742618] d
INFO:starships.planet_obs:Changing excent from pl_orbeccen
-----------
    0.00208 to 0
INFO:starships.planet_obs:It beca

# Build transmission spectrum

In [50]:
"""
param_all: Reduction parameters
telluric fraction to mask (usually varied between 0.2 and 0.5), 
limits for the wings, 
width of the smoothing kernel for the low pass filter (fixed at 51), 
useless param, 
width of the gaussian kernel for low pass filter (fixed at 5),
nPC (nb of principal components) to remove (varied between 1 and 5),
sigma clips params (fixed at 5.0)
"""

#--------------------------------------------------------------------------------------------
# PARAMETERS TO CHANGE

RVsys = [-52.003101]  # change to RV of star system [km/s]
kind_trans = 'transmission'  # emission or transmission
coeffs = [0.02703969,  1.10037972, -0.96372403,  0.28750393]  # limb darkening coefficients
ld_model = 'nonlinear'


tellu_frac = 0.40  # telluric fraction to mask (usually varied between 0.2 and 0.5)
nPC = [0,1]  # number of principal components to remove

mask_wings = 0.97  # fraction of wings of deep tellurics that is masked

#--------------------------------------------------------------------------------------------

cbp = True  # correct bad pixels

iout_all = ['all']
polynome = [False] 
do_tr = [1]
transit_tags = [None]  # use if want to remove spectra (needs to be a list of lists)

kwargs_gen_tr = {
    'coeffs' : coeffs,
    'ld_model' : ld_model,
    'do_tr' : do_tr,
    'kind_trans' : kind_trans,
    'polynome' : polynome,
    'cbp': cbp }

kwargs_build_ts = {
    'clip_ratio' : 6,
    'clip_ts' : 6,
    'unberv_it' : True }

for n_pc in nPC:
    params_all=[[tellu_frac, mask_wings, 51, 41, 5, n_pc, 5.0, 5.0, 5.0, 5.0]]
    list_tr = pl_obs.generate_all_transits(obs, transit_tags, RVsys, params_all, iout_all,
                                           **kwargs_gen_tr, **kwargs_build_ts)

    # Save sequence with all reduction steps
    out_filename = f'sequence_{n_pc}-pc_mask_wings{mask_wings*100:n}'
    pl_obs.save_single_sequences(out_filename, list_tr['1'], path=out_dir,
                                 filename_end=visit_name, save_all=True)
    
    # Save sequence with only the info needed for a retrieval (to compute log likelihood).
    out_filename = f'retrieval_input_{n_pc}-pc_mask_wings{mask_wings*100:n}'
    pl_obs.save_sequences(out_filename, list_tr, do_tr, path=out_dir)

 Masking high variance pixels (quick fix for OH lines).
 Shifting everything in the stellar ref. frame and normalizing by the median.
Spectra 
  Unberv : 74 - 11  
Telluriques 
  Unberv : 74 - 11  
 Masking deep tellurics.
 Building the master out #1.
 74ratio_filt has values <= 0.75!
 Building the transmission spectrum #1.
 Removing the static noise with PCA and sigma cliping.
 Removing the mean.
 Removing the remaining high variance pixels.
 Removing the mean.
 Calculating noise with 0 PCs/scratch/mathisb/HRS_reductions/NIRPS/TRAPPIST-1g/2025-07-08/v001/sequence_0-pc_mask_wings97_data_trs_2025-07-08.npz
/scratch/mathisb/HRS_reductions/NIRPS/TRAPPIST-1g/2025-07-08/v001/retrieval_input_0-pc_mask_wings97_data_info.npz
/scratch/mathisb/HRS_reductions/NIRPS/TRAPPIST-1g/2025-07-08/v001/retrieval_input_0-pc_mask_wings97_data_trs_0.npz
-------------------------

 Masking high variance pixels (quick fix for OH lines).
 Shifting everything in the stellar ref. frame and normalizing by the media

## That's it!

In [51]:
# Verify if every parameter makes sense

print("Mid-transit time (BJD):", "\t", p.mid_tr,
      "\nRadius of the star:", "\t\t", p.R_star.to(u.R_sun),
      "\nRadius of the planet:", "\t\t", p.R_pl.to(u.R_earth),
      "\nObservations in transit:", "\t", obs.iIn,
      "\nObservations out of transit:", "\t", obs.iOut,
      "\nSeparation between planet and star:", "\n", (obs.sep / p.R_star), "R_star"
     )

Mid-transit time (BJD): 	 [2460864.89742618] d 
Radius of the star: 		 [0.1192] solRad 
Radius of the planet: 		 [1.129] earthRad 
Observations in transit: 	 [4 5 6 7 8 9] 
Observations out of transit: 	 [ 0  1  2  3 10 11] 
Separation between planet and star: 
 [1.95946159 1.66404621 1.37273507 1.08884903 0.82015718 0.58789854
 0.45222788 0.49932038 0.69285984 0.94694252 1.22443241 1.51249046] R_star


In [10]:
# # Build master out?
# from starships import transpec

# # function to get master out spectrun for one night
# # try to understand how this function is used in the reduction and put the right arguments there
# flux, master_out, ratio_filter = transpec.build_master_out(wave?, flux_Sref_norm?, obs.iOut, 
#                      kind_lp='filter', box=201, gauss_box=5, master_out=None, kind_mo='median', 
#                      clip_ratio=None, cont=False)

# # then, can I combine the master out for each night to create 1 big master out?